# Description of the Wikidata Property Extraction Script

This script retrieves Wikidata properties using a SPARQL query and exports them to a JSON file.  
It interacts with the Wikidata Query Service, extracts each property's ID (e.g., `P123`) and its English label, and stores the results in a Python dictionary.

After fetching the data, the script:

- Prints the total number of retrieved properties  
- Displays a small preview of the first few entries  
- Saves the full property list to **`wikidata_properties.json`** using a helper function that ensures readable formatting and UTF-8 compatibility


In [ ]:
import json
import requests

def get_all_wikidata_properties():
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "MyReActAgent/0.1 "
    }

    query = """
    SELECT ?property ?propertyLabel WHERE {
      ?property a wikibase:Property .
      SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
    }
    ORDER BY ?property
    """

    response = requests.get(url, params={"query": query}, headers=headers, timeout=20)
    response.raise_for_status()
    data = response.json()

    properties = {}
    for row in data["results"]["bindings"]:
        pid = row["property"]["value"].split("/")[-1]  # extract "P123"
        label = row.get("propertyLabel", {}).get("value", "")
        properties[pid] = label

    return properties


def save_properties_to_json(properties: dict, filename: str):
    """Save the Wikidata properties dictionary to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(
            properties,
            f,
            ensure_ascii=False,   # keep non-English characters
            indent=2              # pretty print
        )
    print(f"Saved {len(properties)} properties to {filename}")


# Usage
all_properties = get_all_wikidata_properties()
print("Found", len(all_properties), "properties.")
print(list(all_properties.items())[:20])  # preview

# Save to file
save_properties_to_json(all_properties, "wikidata_properties.json")